[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/09_ONNX_Graph_Manipulation/01_Inspecting_Models/Inspecting_Models_Deep_Dive.ipynb)

# 9.1 Inspecting ONNX Models — Deep Dive

Graph traversal, programmatic inspection, and structural analysis of ONNX computation graphs.

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Graph Theory Foundations](#section-1) | DAG structure, degree analysis, density |
| 2 | [Graph Traversal Algorithms](#section-2) | BFS, DFS, topological sort, critical path |
| 3 | [Programmatic Inspection API](#section-3) | ModelProto, GraphProto, NodeProto, TensorProto |
| 4 | [Summary Statistics](#section-4) | Op histogram, parameter count, graph geometry |
| 5 | [Subgraph Extraction](#section-5) | Forward/backward slicing, `extract_model` |
| 6 | [Visualization](#section-6) | Netron, Graphviz/DOT, custom summaries |

<a id='section-1'></a>
## Section 1: Graph Theory Foundations for ONNX

Every ONNX model is a **DAG** (Directed Acyclic Graph) $G = (V, E)$ where $V$ = operators and $E$ = tensor data flow.

**Topological ordering** ensures all dependencies execute first:

$$\forall (u,v) \in E: \text{order}(u) < \text{order}(v)$$

**Degree analysis** characterizes node connectivity:
- In-degree $\deg^-(v)$: number of input tensors
- Out-degree $\deg^+(v)$: number of downstream consumers  
- Graph density: $\rho = \frac{|E|}{|V|(|V|-1)}$ — most ONNX graphs are sparse ($\rho \ll 1$)

```
┌──────────────────────────────────────────────────┐
│  [Input X]     [W]   [b]     ONNX DAG Example   │
│      │          │     │                          │
│      ▼    ┌─────┘     │                          │
│   ┌──────────┐        │                          │
│   │   Conv    │ deg⁻=3│                          │
│   └────┬─────┘        │                          │
│        ▼              │                          │
│   ┌──────────┐        │                          │
│   │   Relu    │ deg⁻=1│                          │
│   └────┬─────┘        │                          │
│        ▼              ▼                          │
│   ┌────────┐  ┌─────────┐                       │
│   │ MatMul  │──│   Add   │                       │
│   └────────┘  └────┬────┘                       │
│                     ▼                            │
│               [Output Y]                         │
│   |V|=4  |E|=4  ρ=0.33                          │
└──────────────────────────────────────────────────┘
```

In [ ]:
!pip install onnx onnxruntime numpy matplotlib -q

import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper, shape_inference
from onnx.checker import check_model
from collections import Counter, defaultdict, deque
import matplotlib.pyplot as plt
import os

print(f"ONNX version: {onnx.__version__}")

In [ ]:
def build_multilayer_model():
    """Build a model with skip connection for rich graph analysis."""
    np.random.seed(42)
    inits = [
        numpy_helper.from_array(np.random.randn(16,3,3,3).astype(np.float32)*0.1, "conv1_w"),
        numpy_helper.from_array(np.random.randn(16).astype(np.float32)*0.01, "conv1_b"),
        numpy_helper.from_array(np.random.randn(32,16,3,3).astype(np.float32)*0.1, "conv2_w"),
        numpy_helper.from_array(np.random.randn(32).astype(np.float32)*0.01, "conv2_b"),
        numpy_helper.from_array(np.random.randn(32,16,1,1).astype(np.float32)*0.1, "skip_w"),
        numpy_helper.from_array(np.random.randn(32,10).astype(np.float32)*0.1, "fc_w"),
        numpy_helper.from_array(np.random.randn(10).astype(np.float32)*0.01, "fc_b"),
        numpy_helper.from_array(np.array([1,32], dtype=np.int64), "rs"),
    ]
    nodes = [
        helper.make_node("Conv",["X","conv1_w","conv1_b"],["c1"],kernel_shape=[3,3],pads=[1,1,1,1],name="conv1"),
        helper.make_node("Relu",["c1"],["r1"],name="relu1"),
        helper.make_node("Conv",["r1","conv2_w","conv2_b"],["c2"],kernel_shape=[3,3],pads=[1,1,1,1],name="conv2"),
        helper.make_node("Relu",["c2"],["r2"],name="relu2"),
        helper.make_node("Conv",["r1","skip_w"],["sk"],kernel_shape=[1,1],name="skip_conv"),
        helper.make_node("Add",["r2","sk"],["res"],name="res_add"),
        helper.make_node("Relu",["res"],["r3"],name="relu3"),
        helper.make_node("GlobalAveragePool",["r3"],["gap"],name="gap"),
        helper.make_node("Reshape",["gap","rs"],["flat"],name="reshape"),
        helper.make_node("MatMul",["flat","fc_w"],["mm"],name="fc_mm"),
        helper.make_node("Add",["mm","fc_b"],["Y"],name="fc_add"),
    ]
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1,3,32,32])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1,10])
    g = helper.make_graph(nodes, "resnet_mini", [X], [Y], initializer=inits)
    m = helper.make_model(g, opset_imports=[helper.make_opsetid("",17)])
    m.ir_version = 8
    check_model(m)
    return m

model = build_multilayer_model()
onnx.save(model, "inspection_demo.onnx")
print(f"Model: {len(model.graph.node)} nodes, {len(model.graph.initializer)} initializers")
for i, n in enumerate(model.graph.node):
    print(f"  [{i}] {n.name}: {n.op_type}({', '.join(n.input)}) -> {list(n.output)}")

In [ ]:
def build_graph_adjacency(graph):
    """Build adjacency lists and degree info from an ONNX GraphProto."""
    tp, tc = {}, defaultdict(list)
    for i, node in enumerate(graph.node):
        for out in node.output: tp[out] = i
        for inp in node.input: tc[inp].append(i)
    n = len(graph.node)
    adj, rev = defaultdict(set), defaultdict(set)
    for i, node in enumerate(graph.node):
        for out in node.output:
            for c in tc[out]:
                if c != i:
                    adj[i].add(c); rev[c].add(i)
    in_d = [len(rev[i]) for i in range(n)]
    out_d = [len(adj[i]) for i in range(n)]
    ne = sum(len(adj[i]) for i in range(n))
    return adj, rev, in_d, out_d, ne

adj, rev_adj, in_deg, out_deg, n_edges = build_graph_adjacency(model.graph)
nn = len(model.graph.node)
rho = n_edges / (nn*(nn-1)) if nn > 1 else 0

print(f"=== DAG Structure:  |V|={nn}  |E|={n_edges}  ρ={rho:.4f} ===")
print(f"{'Node':<12} {'Op':<20} {'deg⁻':<6} {'deg⁺':<6} Role")
print("-" * 55)
for i, node in enumerate(model.graph.node):
    role = "source" if in_deg[i]==0 else ("sink" if out_deg[i]==0 else "internal")
    print(f"{node.name:<12} {node.op_type:<20} {in_deg[i]:<6} {out_deg[i]:<6} {role}")

<a id='section-2'></a>
## Section 2: Graph Traversal Algorithms

Both BFS and DFS run in $O(|V| + |E|)$ time.

```
┌─────────────────────────────────────────────────────────┐
│   BFS (Level-Order)            DFS (Depth-First)        │
├─────────────────────────────────────────────────────────┤
│  L0: [Conv1]                  Visit: Conv1→Relu1→Conv2 │
│  L1: [Relu1]                         →Relu2→Add→Relu3  │
│  L2: [Conv2, Skip]                   →GAP→Reshape      │
│  L3: [Relu2]                         →MatMul→Add_fc    │
│  L4: [Add]                    then backtrack to Skip    │
│  L5-L9: Relu3→GAP→...                                  │
│                                                         │
│  Explores breadth first.      Follows depth first.      │
└─────────────────────────────────────────────────────────┘
```

In [ ]:
def bfs_levels(graph, adj, in_deg):
    """BFS level-order traversal from source nodes."""
    n = len(graph.node)
    sources = [i for i in range(n) if in_deg[i] == 0]
    visited = set(sources)
    queue = deque((s, 0) for s in sources)
    result = []
    while queue:
        idx, lv = queue.popleft()
        result.append((lv, idx))
        for nb in sorted(adj[idx]):
            if nb not in visited:
                visited.add(nb)
                queue.append((nb, lv+1))
    return result

bfs = bfs_levels(model.graph, adj, in_deg)
print("=== BFS Level-Order ===")
cur = -1
for lv, idx in bfs:
    if lv != cur:
        print(f"  Level {lv}:", end="")
        cur = lv
    print(f"  {model.graph.node[idx].name}", end="")
    if idx == bfs[-1][1] or bfs[bfs.index((lv,idx))+1][0] != lv:
        print()

wpl = Counter(l for l,_ in bfs)
print(f"\n  Depth: {max(l for l,_ in bfs)+1}  Max width: {max(wpl.values())}")

In [ ]:
def dfs_backward(graph, rev_adj, out_deg):
    """DFS backward reachability from sink nodes (reverse-post-order)."""
    n = len(graph.node)
    sinks = [i for i in range(n) if out_deg[i] == 0]
    visited, post = set(), []
    def _dfs(idx, d):
        visited.add(idx)
        for p in sorted(rev_adj[idx]):
            if p not in visited: _dfs(p, d+1)
        post.append((idx, d))
    for s in sinks:
        if s not in visited: _dfs(s, 0)
    return list(reversed(post))

dfs = dfs_backward(model.graph, rev_adj, out_deg)
print("=== DFS Backward Reachability ===")
for rank, (idx, d) in enumerate(dfs):
    n = model.graph.node[idx]
    print(f"  rank={rank:<3} {n.name:<12} {n.op_type:<18} (depth={d})")

### Topological Sort — Kahn's Algorithm

Repeatedly extract nodes with in-degree zero. Complexity: $O(|V| + |E|)$.

```
┌────────────────────────────────────────────────────────┐
│  Step 1: deg⁻=0 → Queue=[Conv1]                      │
│  Step 2: Remove Conv1 → Queue=[Relu1]                 │
│  Step 3: Remove Relu1 → Queue=[Conv2, Skip]           │
│  Step 4: Continue until Queue empty                    │
│  If |Order| < |V|, graph has a cycle (invalid ONNX).  │
└────────────────────────────────────────────────────────┘
```

In [ ]:
def kahn_topo_sort(graph, adj, in_degree):
    """Kahn's algorithm. Returns (order, is_dag, trace)."""
    n = len(graph.node)
    deg = list(in_degree)
    queue = deque(i for i in range(n) if deg[i] == 0)
    order, trace = [], []
    while queue:
        v = queue.popleft()
        order.append(v)
        edges = []
        for w in sorted(adj[v]):
            deg[w] -= 1
            edges.append((v, w, deg[w]))
            if deg[w] == 0: queue.append(w)
        trace.append((v, list(queue), edges))
    return order, len(order)==n, trace

topo, is_dag, trace = kahn_topo_sort(model.graph, adj, in_deg)
print(f"=== Kahn's Topological Sort (valid DAG: {is_dag}) ===")
for rank, idx in enumerate(topo):
    print(f"  topo({model.graph.node[idx].name}) = {rank}")

print("\n  Trace (first 3 steps):")
for step, (rm, q, edges) in enumerate(trace[:3]):
    qn = [model.graph.node[i].name for i in q]
    print(f"    Step {step}: Remove '{model.graph.node[rm].name}', Queue={qn}")
    for s,d,nd in edges:
        print(f"      ({model.graph.node[s].name}→{model.graph.node[d].name}): deg⁻={nd}")

In [ ]:
def critical_path(graph, adj, in_deg):
    """Longest path (unit cost). T_critical = max_path Σ T_v."""
    n = len(graph.node)
    order, _, _ = kahn_topo_sort(graph, adj, in_deg)
    dist, pred = [0]*n, [-1]*n
    for v in order:
        for w in adj[v]:
            if dist[v]+1 > dist[w]:
                dist[w] = dist[v]+1
                pred[w] = v
    end = max(range(n), key=lambda i: dist[i])
    path = []
    cur = end
    while cur != -1:
        path.append(cur); cur = pred[cur]
    path.reverse()
    return dist[end]+1, path

cp_len, cp = critical_path(model.graph, adj, in_deg)
print(f"=== Critical Path: {cp_len} nodes ===")
for i, idx in enumerate(cp):
    nd = model.graph.node[idx]
    arrow = " → " if i < len(cp)-1 else " ⟹ OUT"
    print(f"  [{i}] {nd.name} ({nd.op_type}){arrow}")

off_cp = sorted(set(range(nn)) - set(cp))
print(f"\n  Non-critical (parallel): {[model.graph.node[i].name for i in off_cp]}")
print(f"  Parallelism: {nn} ops / {cp_len} steps = {nn/cp_len:.2f}x")

<a id='section-3'></a>
## Section 3: Programmatic Inspection API

```
ModelProto
├── ir_version          : int64
├── opset_import[]      : {domain, version}
├── producer_name       : string
└── graph               : GraphProto
    ├── node[]          : NodeProto
    │   ├── op_type, name, domain
    │   ├── input[], output[]   (tensor names)
    │   └── attribute[]         (kernel_shape, pads, etc.)
    ├── input[]         : ValueInfoProto {name, dtype, shape}
    ├── output[]        : ValueInfoProto
    ├── initializer[]   : TensorProto {name, dims, data_type, raw_data}
    └── value_info[]    : ValueInfoProto (intermediates)
```

| Enum | Type | Bytes |  | Enum | Type | Bytes |
|------|------|-------|--|------|------|-------|
| 1 | FLOAT | 4 |  | 7 | INT64 | 8 |
| 6 | INT32 | 4 |  | 10 | FLOAT16 | 2 |
| 11 | DOUBLE | 8 |  | 16 | BFLOAT16 | 2 |

In [ ]:
model = onnx.load("inspection_demo.onnx")
g = model.graph

print("=== ModelProto ===")
print(f"  IR Version: {model.ir_version}")
for op in model.opset_import:
    print(f"  Opset: {op.domain or 'ai.onnx'} v{op.version}")

print(f"\n=== GraphProto ===")
print(f"  Name: {g.name}  |  Nodes: {len(g.node)}  |  Inits: {len(g.initializer)}")
print(f"  Inputs: {[i.name for i in g.input]}")
print(f"  Outputs: {[o.name for o in g.output]}")

print(f"\n=== NodeProto Detail (conv1) ===")
n0 = g.node[0]
print(f"  op_type: {n0.op_type}  name: {n0.name}  domain: {n0.domain or 'ai.onnx'}")
print(f"  inputs: {list(n0.input)}  outputs: {list(n0.output)}")
for a in n0.attribute:
    val = list(a.ints) if a.type == 7 else (a.i if a.type == 2 else a.f)
    print(f"  attr: {a.name} = {val}")

In [ ]:
DTYPE_SZ = {TensorProto.FLOAT:4, TensorProto.DOUBLE:8, TensorProto.FLOAT16:2,
            TensorProto.INT32:4, TensorProto.INT64:8, TensorProto.INT8:1, TensorProto.UINT8:1}

print("=== TensorProto (Initializers) ===")
print(f"{'Name':<10} {'Shape':<16} {'DType':<8} {'Elems':<10} {'Bytes':<10} {'min':>7} {'max':>7}")
print("-" * 72)
tb, te = 0, 0
for init in g.initializer:
    sh = list(init.dims)
    ne = int(np.prod(sh)) if sh else 1
    bs = ne * DTYPE_SZ.get(init.data_type, 4)
    tb += bs; te += ne
    arr = numpy_helper.to_array(init)
    print(f"{init.name:<10} {str(sh):<16} {TensorProto.DataType.Name(init.data_type):<8} "
          f"{ne:<10,} {bs:<10,} {arr.min():>7.3f} {arr.max():>7.3f}")
print(f"\nTotal: {te:,} elements, {tb:,} bytes ({tb/1024:.1f} KB)")

<a id='section-4'></a>
## Section 4: Summary Statistics

**Parameter count:** $P = \sum_{i} \prod_{j} \text{dim}_j(\text{init}_i)$

**Model size:** $\text{Size} = \sum_{i} \left(\prod_{j} \text{dim}_j(\text{init}_i)\right) \times \text{sizeof}(\text{dtype}_i)$

**Graph geometry:**
- Depth = critical path length (min sequential steps)
- Width = max nodes at any BFS level (parallelism potential)
- Density $\rho = |E| / (|V|(|V|-1))$

In [ ]:
op_counts = Counter(n.op_type for n in g.node)
print("=== Op Histogram ===")
for op, cnt in op_counts.most_common():
    print(f"  {op:<20} {cnt:>3}  {'█'*cnt*5}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ops, cnts = zip(*op_counts.most_common())
colors = plt.cm.Set2(np.linspace(0, 1, len(ops)))
bars = axes[0].barh(ops[::-1], cnts[::-1], color=colors, edgecolor="black")
axes[0].set_xlabel("Count"); axes[0].set_title("Operator Frequency", fontweight="bold")
axes[0].bar_label(bars, padding=3, fontweight="bold")
axes[1].pie(cnts, labels=ops, colors=colors, autopct="%1.0f%%", startangle=90)
axes[1].set_title("Op Distribution", fontweight="bold")
plt.tight_layout(); plt.savefig("op_histogram.png", dpi=100, bbox_inches="tight"); plt.show()

In [ ]:
total_p = sum(int(np.prod(list(i.dims))) for i in g.initializer if list(i.dims))
total_b = sum(int(np.prod(list(i.dims)))*DTYPE_SZ.get(i.data_type,4) for i in g.initializer if list(i.dims))

print("=== Parameter Count: P = Σᵢ Πⱼ dim_j(init_i) ===")
print(f"{'Name':<10} {'Shape':<16} {'Params':<12} % Total")
print("-" * 50)
for init in sorted(g.initializer, key=lambda x: -int(np.prod(list(x.dims)))):
    np_ = int(np.prod(list(init.dims)))
    print(f"{init.name:<10} {str(list(init.dims)):<16} {np_:<12,} {100*np_/total_p:.1f}%")
print(f"\n  Total: {total_p:,} params,  {total_b/1024:.1f} KB")

In [ ]:
def graph_geometry(graph, adj, in_deg):
    """Compute depth, width profile, and density."""
    n = len(graph.node)
    sources = [i for i in range(n) if in_deg[i] == 0]
    level = [-1]*n
    q = deque()
    for s in sources: level[s] = 0; q.append(s)
    while q:
        v = q.popleft()
        for w in adj[v]:
            if level[w] < level[v]+1:
                level[w] = level[v]+1; q.append(w)
    depth = max(level)+1
    wp = Counter(level)
    ne = sum(len(adj[i]) for i in range(n))
    return depth, wp, max(wp.values()), ne/(n*(n-1)) if n>1 else 0, level

adj, rev_adj, in_deg, out_deg, _ = build_graph_adjacency(model.graph)
depth, wp, mw, dens, levels = graph_geometry(model.graph, adj, in_deg)

print(f"=== Graph Geometry:  Depth={depth}  MaxWidth={mw}  ρ={dens:.4f} ===")
for lv in sorted(wp.keys()):
    ns = [model.graph.node[i].name for i in range(len(model.graph.node)) if levels[i]==lv]
    print(f"  Level {lv:>2}: {wp[lv]:>2}  {'█'*wp[lv]*4}  {ns}")
shape = "Sequential" if depth > 2*mw else ("Parallel" if mw > 2*depth else "Balanced")
print(f"  Shape: {shape}")

<a id='section-5'></a>
## Section 5: Subgraph Extraction

- **Forward slice**: all nodes reachable *downstream* from a tensor
- **Backward slice**: all nodes required to *compute* a tensor

```
┌────────────────────────────────────────────────────────────┐
│  Full:  [Conv1]→[Relu1]→[Conv2]→[Relu2]→[Add]→...→[FC] │
│                    └→[Skip]──────────→ ┘                  │
│                                                            │
│  Fwd slice from r1: everything after Relu1                │
│  Bwd slice to r2:   Conv1 → Relu1 → Conv2 → Relu2       │
└────────────────────────────────────────────────────────────┘
```

In [ ]:
def forward_slice(graph, tensor):
    """All nodes reachable downstream from tensor."""
    tc = defaultdict(list)
    for i, nd in enumerate(graph.node):
        for inp in nd.input: tc[inp].append(i)
    vis = set(tc[tensor])
    q = deque(vis)
    while q:
        ni = q.popleft()
        for out in graph.node[ni].output:
            for c in tc[out]:
                if c not in vis: vis.add(c); q.append(c)
    return sorted(vis)

def backward_slice(graph, tensor):
    """All nodes required to compute tensor."""
    tp = {out: i for i, nd in enumerate(graph.node) for out in nd.output}
    vis = set()
    q = deque([tp[tensor]] if tensor in tp else [])
    if q: vis.add(q[0])
    while q:
        ni = q.popleft()
        for inp in graph.node[ni].input:
            if inp in tp and tp[inp] not in vis:
                vis.add(tp[inp]); q.append(tp[inp])
    return sorted(vis)

fwd = forward_slice(g, "r1")
print(f"=== Forward Slice from 'r1': {len(fwd)}/{len(g.node)} nodes ===")
for i in fwd: print(f"  [{i}] {g.node[i].name} ({g.node[i].op_type})")

bwd = backward_slice(g, "r2")
print(f"\n=== Backward Slice to 'r2': {len(bwd)}/{len(g.node)} nodes ===")
for i in bwd: print(f"  [{i}] {g.node[i].name} ({g.node[i].op_type})")

In [ ]:
def extract_submodel(model, input_names, output_names):
    """Extract submodel via backward traversal, maintaining graph consistency."""
    graph = model.graph
    pm = {out: nd for nd in graph.node for out in nd.output}
    req, vis = [], set()
    q = deque(output_names)
    while q:
        t = q.popleft()
        if t in vis: continue
        vis.add(t)
        if t in pm:
            nd = pm[t]
            if nd not in req: req.append(nd)
            for inp in nd.input:
                if inp not in vis and inp not in input_names: q.append(inp)
    oo = {id(n): i for i, n in enumerate(graph.node)}
    req.sort(key=lambda n: oo[id(n)])
    ni = [i for i in graph.initializer if i.name in vis]
    new_in = [next((gi for gi in graph.input if gi.name==nm),
              helper.make_tensor_value_info(nm, TensorProto.FLOAT, None)) for nm in input_names]
    new_out = [helper.make_tensor_value_info(nm, TensorProto.FLOAT, None) for nm in output_names]
    sg = helper.make_graph(req, graph.name+"_sub", new_in, new_out, initializer=ni)
    sm = helper.make_model(sg, opset_imports=list(model.opset_import))
    sm.ir_version = model.ir_version
    return sm

sub = extract_submodel(model, ["X"], ["r2"])
print(f"=== Extracted Submodel ===")
print(f"  Nodes: {len(sub.graph.node)} (was {len(g.node)})  Inits: {len(sub.graph.initializer)}")
for n in sub.graph.node: print(f"    {n.name}: {n.op_type}")
onnx.save(sub, "submodel.onnx")

try:
    inf = shape_inference.infer_shapes(model)
    onnx.save(inf, "demo_inf.onnx")
    onnx.utils.extract_model("demo_inf.onnx", "extracted.onnx", ["r1"], ["r3"])
    ext = onnx.load("extracted.onnx")
    print(f"\n  onnx.utils.extract_model: {len(ext.graph.node)} nodes → {[n.op_type for n in ext.graph.node]}")
except Exception as e:
    print(f"\n  extract_model: {e}")

<a id='section-6'></a>
## Section 6: Visualization

| Tool | Best For |
|------|----------|
| **Netron** ([netron.app](https://netron.app)) | Interactive graph exploration |
| **Graphviz/DOT** | Programmatic pipeline diagrams |
| **Custom print** | CI/CD logs, terminal summaries |

In [ ]:
def onnx_to_dot(graph, title="ONNX"):
    """Convert ONNX GraphProto to Graphviz DOT format."""
    lines = [f'digraph "{title}" {{', '  rankdir=TB;',
             '  node [shape=box,style="filled,rounded",fontname=Helvetica];','']
    inames = {i.name for i in graph.initializer}
    for inp in graph.input:
        if inp.name not in inames:
            lines.append(f'  "{inp.name}" [shape=ellipse,fillcolor="#B3E5FC",label="In:{inp.name}"];')
    for out in graph.output:
        lines.append(f'  "{out.name}" [shape=ellipse,fillcolor="#C8E6C9",label="Out:{out.name}"];')
    clr = {"Conv":"#FFCDD2","Relu":"#FFF9C4","Add":"#E1BEE7","MatMul":"#FFE0B2",
           "Reshape":"#B2DFDB","GlobalAveragePool":"#D1C4E9"}
    pm = {}
    for i, nd in enumerate(graph.node):
        c = clr.get(nd.op_type,"#E0E0E0")
        lines.append(f'  "n{i}" [fillcolor="{c}",label="{nd.name}\n({nd.op_type})"];')
        for out in nd.output: pm[out] = f"n{i}"
    for inp in graph.input:
        if inp.name not in inames: pm[inp.name] = inp.name
    for i, nd in enumerate(graph.node):
        for inp in nd.input:
            if inp in pm: lines.append(f'  "{pm[inp]}" -> "n{i}" [label="{inp}",fontsize=8];')
    for out in graph.output:
        if out.name in pm: lines.append(f'  "{pm[out.name]}" -> "{out.name}";')
    lines.append("}"); return "\n".join(lines)

dot = onnx_to_dot(g, "ResNet-Mini")
with open("model_graph.dot","w") as f: f.write(dot)
print("Saved: model_graph.dot")
try:
    import subprocess
    subprocess.run(["dot","-Tpng","model_graph.dot","-o","model_graph.png"], check=True, capture_output=True)
    print("Rendered: model_graph.png")
except (FileNotFoundError, subprocess.CalledProcessError):
    print("Graphviz not found — paste DOT into https://dreampuf.github.io/GraphvizOnline/")

print("\n=== Reference Visualizations ===")
for img in ["assets/dot_linreg.png", "assets/dot_scan_py.png"]:
    if os.path.exists(img):
        from IPython.display import Image, display
        print(f"\n{img}:"); display(Image(filename=img, width=500))
    else:
        print(f"  {img}")

In [ ]:
def model_summary(model):
    """Comprehensive box-formatted model summary."""
    if isinstance(model, str): model = onnx.load(model)
    g = model.graph
    def shp(vi):
        tt = vi.type.tensor_type
        return [d.dim_value if d.HasField("dim_value") else d.dim_param or "?"
                for d in tt.shape.dim] if tt.HasField("shape") else None
    tp = sum(int(np.prod(list(i.dims))) for i in g.initializer if list(i.dims))
    tb = sum(int(np.prod(list(i.dims)))*DTYPE_SZ.get(i.data_type,4) for i in g.initializer if list(i.dims))
    al,_,ind,_,ne = build_graph_adjacency(g)
    dep,_,mw,dns,_ = graph_geometry(g, al, ind)
    cl,cp = critical_path(g, al, ind)
    W = 56
    ln = lambda l,v: f"║  {l:<13}: {str(v):<{W-17}}║"
    print("╔"+"═"*W+"╗")
    print(f"║{'MODEL SUMMARY':^{W}}║")
    print("╠"+"═"*W+"╣")
    for l,v in [("Graph",g.name),("IR Version",model.ir_version),
                ("Opset",f"{model.opset_import[0].domain or 'ai.onnx'} v{model.opset_import[0].version}")]:
        print(ln(l,v))
    print("╠"+"═"*W+"╣")
    for l,v in [("Nodes",len(g.node)),("Edges",ne),("Parameters",f"{tp:,}"),("Size",f"{tb/1024:.1f} KB")]:
        print(ln(l,v))
    print("╠"+"═"*W+"╣")
    for l,v in [("Depth",dep),("Max Width",mw),("Density",f"{dns:.4f}"),("Critical Path",f"{cl} nodes")]:
        print(ln(l,v))
    print("╠"+"═"*W+"╣")
    inames = {i.name for i in g.initializer}
    for inp in g.input:
        if inp.name not in inames:
            print(f"║  IN  {inp.name}: {shp(inp):<{W-8}}║")
    for out in g.output:
        print(f"║  OUT {out.name}: {shp(out):<{W-8}}║")
    print("╠"+"═"*W+"╣")
    for op, cnt in Counter(n.op_type for n in g.node).most_common():
        bar = "█"*min(cnt*3,15)
        print(f"║  {op:<15}{cnt:>3}  {bar:<{W-22}}║")
    print("╚"+"═"*W+"╝")

model_summary(model)

---
## Summary

| Concept | Formula / Technique |
|---------|--------------------|
| DAG structure | $G = (V, E)$ with topological ordering |
| Topological order | $\text{topo}(u) < \text{topo}(v) \iff (u,v) \in E$ |
| Critical path | $T_{\text{critical}} = \max_{\text{path } p} \sum_{v \in p} T_v$ |
| Graph density | $\rho = \frac{|E|}{|V|(|V|-1)}$ |
| Parameter count | $P = \sum_{i} \prod_{j} \text{dim}_j(\text{init}_i)$ |
| BFS / DFS | $O(|V| + |E|)$ time |

### Workflow

1. **Load** model → inspect `ModelProto` metadata
2. **Build** adjacency structures for graph analysis
3. **Traverse** via BFS (level-order) or DFS (dependency chains)
4. **Compute** statistics: op histogram, parameters, depth/width
5. **Extract** subgraphs for focused debugging
6. **Visualize** with Netron (interactive) or DOT (programmatic)

### Reference Visualizations

![Linear Regression Graph](assets/dot_linreg.png)
![Python Scan Model](assets/dot_scan_py.png)

---
**Next:** [Inspecting Models — Apply](./Inspecting_Models_Apply.ipynb) | [Modifying Graphs](../02_Modifying_Graphs/)

In [ ]:
for f in ["inspection_demo.onnx","demo_inf.onnx","submodel.onnx","extracted.onnx",
          "model_graph.dot","model_graph.png","op_histogram.png","param_count.png"]:
    if os.path.exists(f): os.remove(f); print(f"Removed: {f}")
print("Done.")